# Stock Volatility Regression

**Tabular Regression Project** — Predict 30-day realized stock volatility from market and fundamental features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `volatility_30d`

## 2 · Project Overview

We predict **30-day realized stock volatility** from market indicators, fundamental ratios, and macro features. Volatility forecasting is central to risk management, options pricing, and portfolio construction.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given a stock's market cap, P/E ratio, beta, trading volume, debt-to-equity ratio, sector, S&P 500 return, VIX level, earnings surprise, and short interest, predict its **30-day realized volatility** (annualized %).

## 5 · Why This Project Matters

- **Risk management** depends on accurate volatility forecasts for VaR and portfolio hedging.
- **Options pricing** (Black-Scholes) requires volatility as a key input.
- Understanding volatility drivers helps **portfolio construction** and factor investing.
- **Short interest** and **earnings surprises** are actionable signals for volatility traders.
- A core quantitative finance problem with real-world alpha potential.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | 10 (market cap, P/E, beta, volume, D/E, sector, S&P return, VIX, earnings surprise, short interest) |
| **Target** | `volatility_30d` (continuous, annualized %) |
| **Categorical** | sector (6 levels) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "volatility_30d"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 8000
market_cap_b = np.random.lognormal(2, 1.5, n).clip(0.1, 2000)
pe_ratio = np.random.uniform(5, 80, n)
beta = np.random.uniform(0.3, 2.5, n)
avg_volume_m = np.random.lognormal(1, 1.2, n).clip(0.01, 500)
debt_to_equity = np.random.uniform(0, 5, n)
sector = np.random.choice(["tech", "finance", "healthcare", "energy",
                            "consumer", "industrial"], n)
sp500_return = np.random.normal(0.005, 0.03, n)
vix = np.random.uniform(10, 45, n)
earnings_surprise_pct = np.random.normal(0, 5, n)
short_interest_pct = np.random.uniform(0, 30, n)

vol = (beta * 8 + vix * 0.4 + short_interest_pct * 0.3
       + debt_to_equity * 2 - np.log1p(market_cap_b) * 1.5
       + np.abs(earnings_surprise_pct) * 0.5
       + np.abs(sp500_return) * 50
       + np.random.normal(0, 3, n))
vol = np.maximum(vol, 2).clip(2, 80)

df = pd.DataFrame({
    "market_cap_billion": market_cap_b, "pe_ratio": pe_ratio,
    "beta": beta, "avg_daily_volume_m": avg_volume_m,
    "debt_to_equity": debt_to_equity, "sector": sector,
    "sp500_return_1m": sp500_return, "vix": vix,
    "earnings_surprise_pct": earnings_surprise_pct,
    "short_interest_pct": short_interest_pct,
    "volatility_30d": vol
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["beta", "vix", "short_interest_pct",
                          "debt_to_equity", "market_cap_billion", "earnings_surprise_pct"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["volatility_30d"], alpha=0.2, s=8)
    ax.set_xlabel(col); ax.set_ylabel("volatility_30d"); ax.set_title(f"Vol vs {col}")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
df.groupby("sector")["volatility_30d"].mean().sort_values().plot(
    kind="barh", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Mean 30-Day Volatility by Sector"); ax.set_xlabel("Volatility (%)")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `volatility_30d`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel("Volatility (%)")
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: {df[TARGET].mean():.2f}%, Median: {df[TARGET].median():.2f}%")
print(f"Std: {df[TARGET].std():.2f}%, Range: {df[TARGET].min():.2f} - {df[TARGET].max():.2f}%")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

X_train["log_market_cap"] = np.log1p(X_train["market_cap_billion"])
X_test["log_market_cap"] = np.log1p(X_test["market_cap_billion"])

X_train["abs_earnings_surprise"] = np.abs(X_train["earnings_surprise_pct"])
X_test["abs_earnings_surprise"] = np.abs(X_test["earnings_surprise_pct"])

X_train["leverage_risk"] = X_train["beta"] * X_train["debt_to_equity"]
X_test["leverage_risk"] = X_test["beta"] * X_test["debt_to_equity"]

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Beta** is the strongest individual predictor — high-beta stocks are more volatile by definition.
- **VIX** captures market-wide fear — when VIX rises, individual stock volatility increases.
- **Short interest** signals bearish sentiment and creates squeeze risk, both increasing volatility.
- **Market cap** has a negative log-relationship — larger companies are generally less volatile.
- **Earnings surprise magnitude** (not direction) drives event-driven volatility.

**Business takeaway:** For risk management, monitor VIX + beta as the primary volatility drivers. For event-driven trading, earnings surprise magnitude is a strong short-term signal.

## 26 · Limitations

1. Synthetic data — real volatility clustering (GARCH effects) is not captured.
2. No time series structure — past volatility is the best predictor of future volatility.
3. Single-period snapshot — real models update continuously.
4. No options-implied volatility — the market's own forecast is a strong signal.
5. Sector is coarse — sub-industries have very different volatility profiles.

## 27 · How to Improve This Project

1. Add lagged realized volatility (t-1, t-5, t-20) as features.
2. Include implied volatility from options markets.
3. Use GARCH or HAR-RV models for time-series volatility forecasting.
4. Add news sentiment scores and event calendars.
5. Model at intraday frequency for high-frequency risk management.

## 28 · Production Considerations

- Update volatility estimates daily with latest market data.
- Integrate with risk management systems for real-time VaR.
- Monitor model accuracy during regime changes (crisis vs. calm).
- Ensemble with GARCH-family models for robustness.

## 29 · Common Mistakes

1. Ignoring volatility clustering — past volatility is the strongest predictor.
2. Using raw market cap instead of log-transformed.
3. Not distinguishing positive vs. negative earnings surprises (magnitude matters for vol).
4. Training on a calm-market period and deploying in a crisis.
5. Confusing implied volatility (forward-looking) with realized volatility (backward-looking).

## 30 · Mini Challenge / Exercises

1. Remove `beta` — how much does R² drop? Is VIX a partial substitute?
2. Add `beta × vix` interaction — does it capture market regime effects?
3. Try log(volatility_30d) as the target — does it improve normality?
4. Train only on tech stocks — does the model generalize to energy?
5. What VIX level corresponds to the average stock volatility doubling?

## 31 · Final Summary / Key Takeaways

1. **Beta and VIX** are the top volatility predictors.
2. **Short interest** and **earnings surprise magnitude** add event-driven signal.
3. **Log market cap** captures the size-volatility inverse relationship.
4. **Gradient boosting** handles the non-linear interactions well.
5. **Lagged volatility** would be the single biggest improvement for production.
6. **Regime awareness** (crisis vs. calm) is critical for real deployment.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))